# 10 WLASL2000 Video Inference — Auto Understanding Mode

## What this notebook does

This notebook tests the deployed WLASL2000 model on a local video file.

Pipeline:

```text
video file
→ MediaPipe keypoint extraction
→ 60-frame sliding windows
→ WLASL2000 Top-5 prediction
→ confidence + stability rules
→ final best guess / uncertainty output
```

This is designed for a real scenario where you want to understand another deaf/mute signer without knowing sign language yourself.

# 1. Import libraries

In [ ]:
from pathlib import Path
from collections import Counter, deque
import json, time, warnings

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)

# 2. Load deployment config

In [ ]:
PROJECT_ROOT = Path("E:/Be_My_Ear")

DEPLOY_DIR = PROJECT_ROOT / "app" / "models" / "ASL" / "WLASL2000"
CONFIG_FILE = DEPLOY_DIR / "wlasl2000_deployment_config.json"

with open(CONFIG_FILE, "r", encoding="utf-8") as f:
    config = json.load(f)

MODEL_PATH = Path(config["model_path"])
LABEL_MAP_PATH = Path(config["label_map_path"])

with open(LABEL_MAP_PATH, "r", encoding="utf-8") as f:
    raw_label_map = json.load(f)

id_to_gloss = {int(k): v["gloss"] for k, v in raw_label_map.items()}
rules = config["confidence_rules"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Config:", CONFIG_FILE)
print("Model:", MODEL_PATH)
print("Label map:", LABEL_MAP_PATH)
print("Device:", device)
print("Confidence rules:")
print(json.dumps(rules, indent=4))

# 3. Load normalisation stats

In [ ]:
def find_norm_stats_file():
    candidates = []

    if "norm_stats_path" in config:
        candidates.append(Path(config["norm_stats_path"]))

    selected_name = config.get("selected_model_name", "").lower()
    model_dir = PROJECT_ROOT / "models" / "ASL" / "WLASL2000"

    if "light v3" in selected_name:
        candidates.append(model_dir / "wlasl2000_light_v3_two_stage_finetuned_from_wlasl1000_train_norm_stats.npz")

    if "light v2" in selected_name:
        candidates.append(model_dir / "wlasl2000_light_v2_finetuned_from_wlasl1000_train_norm_stats.npz")

    candidates.extend(sorted(model_dir.glob("*norm_stats*.npz")))

    for p in candidates:
        if p.exists():
            return p

    raise FileNotFoundError("Could not find WLASL2000 norm stats .npz file in models/ASL/WLASL2000.")

NORM_STATS_FILE = find_norm_stats_file()
stats = np.load(NORM_STATS_FILE)
train_mean = stats["mean"].astype(np.float32)
train_std = stats["std"].astype(np.float32)

print("Norm stats:", NORM_STATS_FILE)
print("Mean:", train_mean.shape, "Std:", train_std.shape)

# 4. Load deployed model

In [ ]:
class BiGRUAttentionDeploy(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, num_layers=2, dropout=0.35):
        super().__init__()

        self.input_projection = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.gru = nn.GRU(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )

        bi_hidden = hidden_size * 2

        self.attention = nn.Sequential(
            nn.Linear(bi_hidden, hidden_size),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(bi_hidden, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        x = self.input_projection(x)
        gru_out, _ = self.gru(x)
        scores = self.attention(gru_out).squeeze(-1)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        context = torch.sum(gru_out * weights, dim=1)
        return self.classifier(context)


checkpoint = torch.load(MODEL_PATH, map_location=device)

num_classes = int(config["num_classes"])
input_size = int(config["input_shape"][1])
hidden_size = int(checkpoint.get("hidden_size", 320))
num_layers = int(checkpoint.get("num_layers", 2))
dropout = float(checkpoint.get("dropout", 0.35))

model = BiGRUAttentionDeploy(input_size, hidden_size, num_classes, num_layers, dropout).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Loaded:", config["selected_model_name"])
print("Architecture:", checkpoint.get("architecture", "BiGRUAttentionDeploy"))
print("Classes:", num_classes)

# 5. Setup MediaPipe

In [ ]:
mp_holistic = mp.solutions.holistic

SEQUENCE_LENGTH = int(config["sequence_length"])
BASE_FEATURE_SIZE = int(config["base_keypoint_shape"][1])
INPUT_SIZE = int(config["input_shape"][1])

LEFT_HAND_SIZE = 21 * 3
RIGHT_HAND_SIZE = 21 * 3
POSE_SIZE = 33 * 4
FEATURE_SIZE = LEFT_HAND_SIZE + RIGHT_HAND_SIZE + POSE_SIZE

def extract_landmarks_from_results(results):
    left = np.array([[lm.x, lm.y, lm.z] for lm in results.left_hand_landmarks.landmark], dtype=np.float32).flatten() if results.left_hand_landmarks else np.zeros(LEFT_HAND_SIZE, dtype=np.float32)
    right = np.array([[lm.x, lm.y, lm.z] for lm in results.right_hand_landmarks.landmark], dtype=np.float32).flatten() if results.right_hand_landmarks else np.zeros(RIGHT_HAND_SIZE, dtype=np.float32)
    pose = np.array([[lm.x, lm.y, lm.z, lm.visibility] for lm in results.pose_landmarks.landmark], dtype=np.float32).flatten() if results.pose_landmarks else np.zeros(POSE_SIZE, dtype=np.float32)
    return np.concatenate([left, right, pose]).astype(np.float32)

def process_frame_to_keypoints(frame_bgr, holistic):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    rgb.flags.writeable = False
    results = holistic.process(rgb)
    return extract_landmarks_from_results(results)

print("MediaPipe ready.")
print("Sequence length:", SEQUENCE_LENGTH)
print("Base feature size:", BASE_FEATURE_SIZE)
print("Model input size:", INPUT_SIZE)

# 6. Prediction and decision functions

In [ ]:
def prepare_model_input(keypoint_sequence):
    keypoint_sequence = np.asarray(keypoint_sequence, dtype=np.float32)

    if keypoint_sequence.shape != (SEQUENCE_LENGTH, BASE_FEATURE_SIZE):
        raise ValueError(f"Expected {(SEQUENCE_LENGTH, BASE_FEATURE_SIZE)}, got {keypoint_sequence.shape}")

    normalised = (keypoint_sequence - train_mean.reshape(1, -1)) / (train_std.reshape(1, -1) + 1e-6)

    velocity = np.zeros_like(normalised, dtype=np.float32)
    velocity[1:] = normalised[1:] - normalised[:-1]

    features = np.concatenate([normalised, velocity], axis=1).astype(np.float32)
    return torch.tensor(features, dtype=torch.float32).unsqueeze(0)

def predict_keypoint_sequence(keypoint_sequence, top_k=5):
    x = prepare_model_input(keypoint_sequence).to(device)

    with torch.no_grad():
        logits = model(x)
        probs = F.softmax(logits, dim=1)[0].detach().cpu().numpy()

    top_ids = np.argsort(probs)[-top_k:][::-1]

    top_predictions = [
        {
            "label_id": int(label_id),
            "gloss": id_to_gloss.get(int(label_id), str(label_id)),
            "probability": float(probs[label_id])
        }
        for label_id in top_ids
    ]

    top1 = top_predictions[0]
    top2_prob = top_predictions[1]["probability"] if len(top_predictions) > 1 else 0.0

    return {
        "top1_label_id": top1["label_id"],
        "top1_gloss": top1["gloss"],
        "top1_confidence": top1["probability"],
        "top1_top2_margin": float(top1["probability"] - top2_prob),
        "top_k": top_predictions
    }

def decision_from_prediction(prediction, recent_predictions=None):
    confidence = prediction["top1_confidence"]
    margin = prediction["top1_top2_margin"]

    auto_accept_confidence = float(rules.get("auto_accept_confidence", 0.45))
    uncertain_min_confidence = float(rules.get("uncertain_min_confidence", 0.25))
    required_margin = float(rules.get("top1_top2_margin", 0.05))
    min_repeated = int(rules.get("min_repeated_predictions", 2))
    stability_count = int(rules.get("stability_window_count", 3))

    stable = False

    if recent_predictions is not None and len(recent_predictions) >= min_repeated:
        last_items = list(recent_predictions)[-stability_count:]
        glosses = [item["top1_gloss"] for item in last_items]
        stable = Counter(glosses)[prediction["top1_gloss"]] >= min_repeated
    else:
        stable = True

    if confidence >= auto_accept_confidence and margin >= required_margin and stable:
        return {
            "status": "accepted",
            "stable": stable,
            "message": f"Detected sign: {prediction['top1_gloss']}",
            "top1_gloss": prediction["top1_gloss"],
            "confidence": confidence,
            "margin": margin
        }

    if confidence >= uncertain_min_confidence:
        alternatives = ", ".join([x["gloss"] for x in prediction["top_k"]])
        return {
            "status": "uncertain",
            "stable": stable,
            "message": f"I think this may mean: {prediction['top1_gloss']} | Alternatives: {alternatives}",
            "top1_gloss": prediction["top1_gloss"],
            "confidence": confidence,
            "margin": margin
        }

    return {
        "status": "repeat",
        "stable": stable,
        "message": "I am not sure. Please sign again slowly.",
        "top1_gloss": prediction["top1_gloss"],
        "confidence": confidence,
        "margin": margin
    }

# 7. Set video path

In [ ]:
VIDEO_PATH = Path("E:/Be_My_Ear/test_videos/sample_sign_video.mp4")

print("Video exists:", VIDEO_PATH.exists())
print("Video path:", VIDEO_PATH)

if not VIDEO_PATH.exists():
    print("Change VIDEO_PATH to your real test video before running the next cells.")

# 8. Extract keypoints from video

In [ ]:
def extract_keypoints_from_full_video(video_path, max_frames=None):
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {video_path}")

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    print("Frame count:", frame_count)
    print("FPS:", fps)

    keypoints = []

    with mp_holistic.Holistic(
        static_image_mode=False,
        model_complexity=1,
        enable_segmentation=False,
        refine_face_landmarks=False,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as holistic:
        total = frame_count if max_frames is None else min(frame_count, max_frames)

        for frame_idx in tqdm(range(total), desc="Extracting video keypoints"):
            success, frame = cap.read()

            if not success:
                break

            keypoints.append(process_frame_to_keypoints(frame, holistic))

    cap.release()

    return np.array(keypoints, dtype=np.float32), fps

full_keypoints, fps = extract_keypoints_from_full_video(VIDEO_PATH)
print("Extracted keypoints shape:", full_keypoints.shape)

# 9. Create sliding windows

In [ ]:
def resample_sequence(sequence, target_length=60):
    sequence = np.asarray(sequence, dtype=np.float32)

    if len(sequence) == target_length:
        return sequence

    if len(sequence) == 0:
        return np.zeros((target_length, BASE_FEATURE_SIZE), dtype=np.float32)

    old_x = np.linspace(0, 1, len(sequence))
    new_x = np.linspace(0, 1, target_length)

    resampled = []
    for feature_idx in range(sequence.shape[1]):
        resampled.append(np.interp(new_x, old_x, sequence[:, feature_idx]))

    return np.stack(resampled, axis=1).astype(np.float32)

def create_sliding_windows(keypoints, window_size=60, stride=15):
    windows, meta = [], []
    n = len(keypoints)

    if n <= window_size:
        windows.append(resample_sequence(keypoints, window_size))
        meta.append({"start_frame": 0, "end_frame": max(n - 1, 0)})
        return windows, meta

    for start in range(0, n - window_size + 1, stride):
        end = start + window_size
        windows.append(keypoints[start:end])
        meta.append({"start_frame": start, "end_frame": end - 1})

    if meta[-1]["end_frame"] < n - 1:
        windows.append(keypoints[-window_size:])
        meta.append({"start_frame": n - window_size, "end_frame": n - 1})

    return windows, meta

WINDOW_STRIDE = 15
windows, window_metadata = create_sliding_windows(full_keypoints, SEQUENCE_LENGTH, WINDOW_STRIDE)

print("Windows:", len(windows))
print("First window shape:", windows[0].shape)

# 10. Predict all windows

In [ ]:
recent_predictions = deque(maxlen=int(rules.get("stability_window_count", 3)))
window_results = []

for i, window in enumerate(tqdm(windows, desc="Predicting windows")):
    prediction = predict_keypoint_sequence(window, top_k=int(rules.get("output_top_k", 5)))
    recent_predictions.append(prediction)
    decision = decision_from_prediction(prediction, recent_predictions=recent_predictions)

    window_results.append({
        "window_id": i,
        "start_frame": window_metadata[i]["start_frame"],
        "end_frame": window_metadata[i]["end_frame"],
        "status": decision["status"],
        "stable": decision["stable"],
        "top1_gloss": prediction["top1_gloss"],
        "confidence": prediction["top1_confidence"],
        "margin": prediction["top1_top2_margin"],
        "message": decision["message"],
        "top5_glosses": ", ".join([x["gloss"] for x in prediction["top_k"]]),
        "top5_probabilities": ", ".join([f"{x['probability']:.4f}" for x in prediction["top_k"]])
    })

results_df = pd.DataFrame(window_results)
display(results_df.head(20))

# 11. Summarise video output

In [ ]:
accepted = results_df[results_df["status"] == "accepted"].copy()
uncertain = results_df[results_df["status"] == "uncertain"].copy()

collapsed_signs = []
for gloss in accepted["top1_gloss"].tolist():
    if len(collapsed_signs) == 0 or collapsed_signs[-1] != gloss:
        collapsed_signs.append(gloss)

print("Accepted signs:", collapsed_signs)
print("Accepted windows:", len(accepted))
print("Uncertain windows:", len(uncertain))
print("Repeat windows:", len(results_df[results_df["status"] == "repeat"]))

if collapsed_signs:
    print("\nAuto-understanding output:")
    print(" ".join(collapsed_signs))
elif len(uncertain) > 0:
    best = uncertain.sort_values("confidence", ascending=False).iloc[0]
    print("\nNo stable accepted sign. Best uncertain guess:")
    print(best["message"])
else:
    print("\nModel is not confident. Ask signer to sign again slowly.")

# 12. Save video inference report

In [ ]:
REPORT_DIR = PROJECT_ROOT / "reports" / "inference"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

safe_name = VIDEO_PATH.stem.replace(" ", "_")
report_file = REPORT_DIR / f"{safe_name}_wlasl2000_video_inference.csv"
summary_file = REPORT_DIR / f"{safe_name}_wlasl2000_video_summary.json"

results_df.to_csv(report_file, index=False)

summary = {
    "video_path": str(VIDEO_PATH),
    "model": config["selected_model_name"],
    "accepted_signs": collapsed_signs,
    "auto_understanding_output": " ".join(collapsed_signs),
    "num_windows": int(len(results_df)),
    "num_accepted_windows": int(len(accepted)),
    "num_uncertain_windows": int(len(uncertain)),
    "rules": rules
}

with open(summary_file, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)

print("Saved report:", report_file)
print("Saved summary:", summary_file)